# Migrating to Amazon Bedrock from OpenAI or Anthropic

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

Code written for the OpenAI or Anthropic SDK moves to Amazon Bedrock by changing the
base URL, the key and the model ID. This notebook makes that change for both SDKs on
both endpoints, then handles what a straight port gets wrong: model IDs, URL paths,
sampling parameters and the API each model serves, with a small shim that lets one call
site work across families.

That is the shape of a chat port. A retrieval pipeline also calls `embeddings.create`,
which both endpoints serve as a route and neither serves a model for, so it moves a
different way (section 3).

**Covered:** the OpenAI and Anthropic SDKs on both endpoints, model IDs, one client per
URL path, a compatibility shim, function calling, streaming on the three APIs, LLM
gateways, what you gain and lose, where embeddings go, and a checklist.

## Setup

Install the pinned requirements first (`pip install -r ../requirements.txt`, Python 3.11
or later). Both SDKs authenticate with a short-term Bedrock API key minted from your IAM
credentials;
[`00-foundations/01`](../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb)
explains the key, the endpoints and the paths. From `_shared/bedrock.py`, this
repository's helper module, the notebook takes `api_prefix`, which picks a model's URL
path on each endpoint.

In [1]:
import json
import re
import sys

import anthropic
import openai
from aws_bedrock_token_generator import provide_token
from openai import OpenAI

sys.path.insert(0, "../_shared")
from bedrock import api_prefix, runtime_id_for

REGION = "us-east-1"
MANTLE = f"https://bedrock-mantle.{REGION}.api.aws"
RUNTIME = f"https://bedrock-runtime.{REGION}.amazonaws.com"
KEY = provide_token(region=REGION)  # lasts at most 12 hours; mint a new one before then
QUESTION = "What is the capital of Australia? Answer in one word."
# Checked in every cell that asks QUESTION, so a wrong or empty reply shows as False
# rather than as a line that reads like an answer.
ANSWER = "canberra"

## 1. The OpenAI SDK

For OpenAI code the change is `base_url`, `api_key` and the model ID. `bedrock-runtime`,
which AWS recommends for new applications, serves the OpenAI APIs at `/openai/v1`, and
several families there need an inference profile such as `us.openai.gpt-5.6-sol`.
Sending the `bedrock-mantle` ID `openai.gpt-5.6-sol` to `bedrock-runtime` returns
`Invocation of model ID openai.gpt-5.6-sol with on-demand throughput isn’t supported`,
and an ID that differs between the endpoints returns `The provided model identifier is
invalid`. Choose `bedrock-mantle` when you need something only it has (section 9).

In [2]:
# Before: client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
DESTINATIONS = [
    ("bedrock-runtime", RUNTIME + "/openai/v1", "us.openai.gpt-5.6-sol"),
    ("bedrock-mantle", MANTLE + "/openai/v1", "openai.gpt-5.6-sol"),
]
for label, base_url, model in DESTINATIONS:
    client = OpenAI(base_url=base_url, api_key=KEY, timeout=180, max_retries=5)
    response = client.responses.create(model=model, input=QUESTION, max_output_tokens=2000, store=False)
    text = response.output_text.strip()
    print(f"{label:16} {model:24} -> {text}  (names Canberra: {ANSWER in text.lower()})")

bedrock-runtime  us.openai.gpt-5.6-sol    -> Canberra  (names Canberra: True)


bedrock-mantle   openai.gpt-5.6-sol       -> Canberra  (names Canberra: True)


The same change through environment variables, for code you would rather not edit. The
model ID still has to change, because the variables do not translate it.

```bash
export OPENAI_BASE_URL="https://bedrock-runtime.us-east-1.amazonaws.com/openai/v1"
export OPENAI_API_KEY="$(python -c 'from aws_bedrock_token_generator import provide_token; print(provide_token(region="us-east-1"))')"
```

Code that lists models does not survive the export. `bedrock-runtime` does not
implement the OpenAI-compatible Models API, as the [Chat Completions API](https://docs.aws.amazon.com/bedrock/latest/userguide/inference-chat-completions.html)
page states, so `client.models.list()` against the base URL above returns 404 with
the body `<UnknownOperationException/>`, which the SDK raises as
`openai.NotFoundError`. The same call against `bedrock-mantle` `/v1` listed 61 models
(`us-east-1`, 12 November 2026). That route sits on the bare `/v1` path there whatever
path the model's own inference APIs use, so `GET /openai/v1/models` on
`bedrock-mantle` returns 404 with an empty body. For a Region's catalogue on
`bedrock-runtime`, call the control plane's `ListFoundationModels` and
`ListInferenceProfiles`.

Do not hardcode a long-term key. A short-term key is a presigned SigV4 request that
carries the minting role's permissions and expires within 12 hours.

## 2. The Anthropic SDK

For Anthropic code, point `base_url` at the `/anthropic` path of either endpoint,
without `/v1`, which the SDK adds. Pass the Bedrock key as `auth_token`, which the SDK
sends as a bearer token, with `api_key=None`. On `bedrock-runtime` Claude needs an
inference profile such as `us.anthropic.claude-opus-5`; the reply's `model` field
carries the short model name either way.

In [3]:
for label, base_url, model in (
    ("bedrock-runtime", RUNTIME + "/anthropic", "us.anthropic.claude-opus-5"),
    ("bedrock-mantle", MANTLE + "/anthropic", "anthropic.claude-opus-5"),
):
    claude = anthropic.Anthropic(base_url=base_url, auth_token=KEY, api_key=None, timeout=180, max_retries=5)
    message = claude.messages.create(model=model, max_tokens=200, messages=[{"role": "user", "content": QUESTION}])
    text = "".join(block.text for block in message.content if block.type == "text").strip()
    print(f"{label:16} {model:28} -> {text}   (model field: {message.model}; "
          f"names Canberra: {ANSWER in text.lower()})")

bedrock-runtime  us.anthropic.claude-opus-5   -> Canberra   (model field: claude-opus-5; names Canberra: True)


bedrock-mantle   anthropic.claude-opus-5      -> Canberra   (model field: claude-opus-5; names Canberra: True)


## 3. Model IDs

OpenAI and Anthropic model names do not exist on Bedrock. Map each one to a Bedrock
model before moving traffic. The table below is a starting point by rough capability,
so evaluate each replacement on your own tasks.

| Name in your code | Bedrock model |
|---|---|
| `gpt-4o` | `openai.gpt-5.6-sol` |
| `gpt-4o-mini` | `openai.gpt-5.4` |
| `o1` | `openai.gpt-5.6-sol` |
| `gpt-3.5-turbo` | `openai.gpt-oss-20b` |
| `claude-3-5-sonnet` | `anthropic.claude-sonnet-5` |
| `claude-3-haiku` | `anthropic.claude-haiku-4-5` |

The cell below shows what an OpenAI name returns on Bedrock. For whether a target is
served where you call it, read that model's
[card](https://docs.aws.amazon.com/bedrock/latest/userguide/model-cards.html), which
gives the endpoints and the inference IDs that serve the model.

The table holds chat models, which is what that path serves: `GET /v1/models` on
`bedrock-mantle` in `us-east-1` on 15 October 2026 listed no embedding model at all. An
OpenAI embedding name such as `text-embedding-3-small` therefore has no entry here.
Those calls go to `bedrock-runtime` `InvokeModel` with the provider's own body,
`amazon.titan-embed-text-v2:0` being one starting point, and `ListFoundationModels`
filtered on output modality `EMBEDDING` lists what a Region has. Re-embed the corpus
when you switch model: vectors from two models are not comparable, so a store built with
one cannot be queried with the other (Things that differ).

In [4]:
try:
    OpenAI(base_url=MANTLE + "/openai/v1", api_key=KEY, timeout=60, max_retries=0).responses.create(
        model="gpt-4o", input=QUESTION, max_output_tokens=16, store=False)
except openai.NotFoundError as exc:
    print(f"gpt-4o -> {exc.status_code} {exc.body.get('message') if isinstance(exc.body, dict) else exc.body}")
else:
    # A success here means an OpenAI name now resolves on Bedrock, so the table above is wrong.
    raise RuntimeError("gpt-4o answered; re-check the model IDs above against the model cards")

gpt-4o -> 404 The model 'gpt-4o' does not exist


## 4. One client per URL path

One base URL does not cover every family. On `bedrock-mantle` the hosted GPT models,
Gemma 4 and Grok are at `/openai/v1`, gpt-oss and the other open-weight families at
`/v1`, and Claude at `/anthropic/v1`; on `bedrock-runtime` every OpenAI-compatible model
is at `/openai/v1`. A model has one chat path on `bedrock-mantle`. Sent to the other
one it returns 400 ``model `openai.gpt-oss-120b` isn't supported on this route``, and
a model served on neither, as every Claude model here is, names the API instead
(Things that differ). `api_prefix` picks the path, and the factory below keeps one
client per base URL.

A base URL that is served by no API at all fails differently. On
`bedrock-runtime` the OpenAI APIs live under `/openai/v1`, so a client built on the
bare `https://bedrock-runtime.us-east-1.amazonaws.com/v1` gets HTTP 200 and
`{"Output":{"__type":"com.amazon.coral.service#UnknownOperationException"},"Version":"1.0"}`,
for `/v1/embeddings` and `/v1/chat/completions` alike. The OpenAI SDK turns that body
into `ValueError: No embedding data received` on `embeddings.create`, and on
`chat.completions.create` into a `ChatCompletion` whose `id`, `choices`, `model` and
`usage` are all `None`, so `choices[0]` raises `TypeError: 'NoneType' object is not
subscriptable` several lines away from the wrong base URL (`us-east-1`,
20 October 2026).

The ID changes with the endpoint as well as the path. `bedrock-mantle` takes the plain
model ID, and `bedrock-runtime` takes the inference-profile form, which is not a fixed
prefix away from it: `openai.gpt-5.6-sol` becomes `us.openai.gpt-5.6-sol`, while
`qwen.qwen3-32b` becomes `qwen.qwen3-32b-v1:0` (`us-east-1`, 27 October 2026). The
mantle ID sent to `bedrock-runtime` is refused, in wording that depends on the model:
`openai.gpt-5.6-sol` returns 400 `Invocation of model ID openai.gpt-5.6-sol with
on-demand throughput isn’t supported. Retry your request with the ID or ARN of an
inference profile that contains this model.`, and `qwen.qwen3-32b` returns 400 `The
provided model identifier is invalid.`, which names nothing to look up. So the factory
looks the ID up and returns it beside the client, and the loop sends what it was given.
`runtime_id_for` answers `None` for a model that endpoint does not serve, such as
`google.gemma-4-31b`, and the factory raises there rather than falling back to an ID
that cannot work.
[`00-foundations/01`](../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb)
section 7 covers the profile forms per endpoint.

In [5]:
_clients: dict[str, OpenAI] = {}


def client_for(model: str, endpoint: str = "mantle") -> tuple[OpenAI, str]:
    """A client on the URL path this model needs, and the ID that endpoint takes."""
    base_url = (MANTLE if endpoint == "mantle" else RUNTIME) + api_prefix(model, endpoint)
    if base_url not in _clients:
        _clients[base_url] = OpenAI(base_url=base_url, api_key=KEY, timeout=180, max_retries=5)
    model_id = model
    if endpoint == "runtime":
        # The profile form is not the mantle ID and not a fixed prefix from it, and None
        # means the model is not on this endpoint, so raise instead of sending the mantle ID.
        model_id = runtime_id_for(model, REGION)
        if model_id is None:
            raise ValueError(f"{model} is not served by bedrock-runtime in {REGION}")
    return _clients[base_url], model_id


for model, endpoint in (("openai.gpt-5.6-sol", "mantle"), ("openai.gpt-oss-120b", "mantle"),
                        ("google.gemma-4-31b", "mantle"), ("qwen.qwen3-32b", "mantle"),
                        ("openai.gpt-5.6-sol", "runtime"), ("qwen.qwen3-32b", "runtime")):
    client, model_id = client_for(model, endpoint)
    path = str(client.base_url).removeprefix(MANTLE).removeprefix(RUNTIME)
    ask = [{"role": "user", "content": QUESTION}]
    reply = client.chat.completions.create(model=model_id, messages=ask, max_completion_tokens=2000)
    text = (reply.choices[0].message.content or "").strip()
    print(f"{model:22} {endpoint:8} {path:12} {model_id:24} -> {text}  "
          f"(names Canberra: {ANSWER in text.lower()})")

openai.gpt-5.6-sol     mantle   /openai/v1/  openai.gpt-5.6-sol       -> Canberra  (names Canberra: True)


openai.gpt-oss-120b    mantle   /v1/         openai.gpt-oss-120b      -> Canberra  (names Canberra: True)


google.gemma-4-31b     mantle   /openai/v1/  google.gemma-4-31b       -> Canberra  (names Canberra: True)
qwen.qwen3-32b         mantle   /v1/         qwen.qwen3-32b           -> Canberra  (names Canberra: True)


openai.gpt-5.6-sol     runtime  /openai/v1/  us.openai.gpt-5.6-sol    -> Canberra  (names Canberra: True)


qwen.qwen3-32b         runtime  /openai/v1/  qwen.qwen3-32b-v1:0      -> Canberra  (names Canberra: True)


## 5. A compatibility shim

Sampling parameters and service tiers are accepted per model, so a shim that carries a
table of restrictions goes stale. The service names the parameter it refuses, as in
`Unsupported parameter: 'top_p' is not supported with this model.` or `` `temperature`
is deprecated for this model.``, so the shim below sends only the options you pass, and
when a 400 names one of them, drops it and tries again, once per option, saying what it
dropped. It uses Messages for Claude and Chat Completions for every other model, because
every other family on `bedrock-mantle` serves Chat Completions and accepts
`max_completion_tokens`, as
[`01-choosing-a-model-and-api`](01-choosing-a-model-and-api.ipynb) measures.

It carries sampling options and nothing else, and a `tools` list is not safe to pass
through it. `send` reads only the text part of a reply, so
`complete(model, prompt, tools=[...])` has two outcomes and neither is the one you want.
A model that refuses the OpenAI function shape raises, because the 400 names no option in
`TUNABLE` and `refused_option` answers `None`: `anthropic.claude-sonnet-5` and
`anthropic.claude-haiku-4-5` refuse that shape on Messages, and `openai.gpt-5.6-sol`
refuses function tools on Chat Completions. A model that accepts it answers with a tool
call and no text, and `complete` returns `''` with no error, as `openai.gpt-5.4`,
`openai.gpt-5.5`, `openai.gpt-oss-120b`, `qwen.qwen3-32b` and `google.gemma-4-31b` each
did (`us-east-1`, 12 November 2026). Tools go per family (section 6).

In [6]:
TUNABLE = ("temperature", "top_p", "service_tier")
CLAUDE = anthropic.Anthropic(base_url=MANTLE + "/anthropic", auth_token=KEY, api_key=None,
                             timeout=180, max_retries=5)


def refused_option(exc: Exception) -> str | None:
    """The option in TUNABLE that a 400 names, if it names one."""
    body = exc.body if isinstance(getattr(exc, "body", None), dict) else {}
    error = body.get("error")
    message = str((error.get("message") if isinstance(error, dict) else None) or body.get("message") or "")
    quoted = [name for name in re.findall(r"[`'\"]([a-z_]+)[`'\"]", message) if name in TUNABLE]
    return quoted[0] if quoted else next((name for name in TUNABLE if name in message), None)


def send(model: str, prompt: str, max_tokens: int, options: dict) -> str:
    """One request: Messages for Claude, Chat Completions for every other model."""
    messages = [{"role": "user", "content": prompt}]
    if model.startswith("anthropic."):
        reply = CLAUDE.messages.create(model=model, max_tokens=max_tokens, messages=messages, **options)
        return "".join(block.text for block in reply.content if block.type == "text")
    client, model_id = client_for(model)
    reply = client.chat.completions.create(model=model_id, messages=messages,
                                           max_completion_tokens=max_tokens, **options)
    return reply.choices[0].message.content or ""

In [7]:
def complete(model: str, prompt: str, *, max_tokens: int = 2000, **options) -> str:
    """Send the options you pass; drop each one a 400 names, once, and say so."""
    dropped = []
    for _ in range(len(options) + 1):
        try:
            text = send(model, prompt, max_tokens, options)
        except (openai.BadRequestError, anthropic.BadRequestError) as exc:
            name = refused_option(exc)
            if name not in options:
                raise
            options.pop(name)
            dropped.append(name)
            continue
        if dropped:
            print(f"  [{model}: dropped {', '.join(dropped)}]")
        return text
    raise RuntimeError(f"{model}: still refused after dropping {dropped}")

In [8]:
# The same call site for five models, each passed options some of them refuse.
for model in ("openai.gpt-5.6-sol", "google.gemma-4-31b", "qwen.qwen3-32b",
              "anthropic.claude-haiku-4-5", "anthropic.claude-sonnet-5"):
    answer = complete(model, QUESTION, temperature=0.7, top_p=0.95, service_tier="flex").strip()
    print(f"{model:28} -> {answer}  (names Canberra: {ANSWER in answer.lower()})")

  [openai.gpt-5.6-sol: dropped service_tier, temperature, top_p]
openai.gpt-5.6-sol           -> Canberra  (names Canberra: True)


google.gemma-4-31b           -> Canberra  (names Canberra: True)
qwen.qwen3-32b               -> Canberra  (names Canberra: True)


  [anthropic.claude-haiku-4-5: dropped service_tier, temperature]
anthropic.claude-haiku-4-5   -> Canberra  (names Canberra: True)


  [anthropic.claude-sonnet-5: dropped service_tier, temperature, top_p]
anthropic.claude-sonnet-5    -> **Canberra**  (names Canberra: True)


## 6. Function calling

A `tools` list does not port with the base URL, and the three APIs take three shapes.
Chat Completions takes the nested OpenAI function tool,
`{"type": "function", "function": {"name", "description", "parameters"}}`. The Responses
API takes the same fields flat, `{"type": "function", "name", "description",
"parameters"}`, on both of the paths that serve it: the nested form there returns 400
``invalid request body: Invalid 'tools': missing field `name` ``, and the flat form on
Chat Completions returns 400 ``invalid request body: Invalid 'tools': missing field
`function` ``, with the bare `/v1` path wrapping the same sentence in `Failed to
deserialize the JSON body into the target type`. Messages takes `{"name",
"description", "input_schema"}` with no wrapper, and the OpenAI shape sent to Messages
returns 400 ``tool type 'function' is not supported for this model`` on both Claude
models in section 5 (`us-east-1`, 12 November 2026).

`openai.gpt-5.6-sol` needs one more decision. The Chat Completions request below, which
`openai.gpt-5.4` and `openai.gpt-5.5` answer with a tool call, returns 400 `Function
tools with reasoning_effort are not supported for gpt-5.6-sol in
/v1/chat/completions. To use function tools, use /v1/responses or set reasoning_effort
to 'none'.` for that model, with no `reasoning_effort` in the request. Sending
`reasoning_effort="none"` makes the same request answer, and so does the Responses API,
which leaves the reasoning effort alone; the cell below takes the Responses route. Read
the path in that advice as the route the service matched: `/v1/responses` on
`bedrock-mantle` returns 400 `The model 'openai.gpt-5.6-sol' does not support the
'/v1/responses' API`, and the model is served at `/openai/v1/responses`, which
`api_prefix` returns (`us-east-1`, 12 November 2026). Its [model
card](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-openai-gpt-56-sol.html)
lists both APIs on that endpoint and says nothing either way about function tools.

The cell sends one tool definition to four models and reads back what each asks for.
Each API names the stop differently: `tool_calls` on Chat Completions, `completed` on
the Responses API, and `tool_use` on Messages. `completed` is the status of the whole
response rather than a tool signal, so unlike the other two it does not by itself say a
tool was requested; the cell raises when a reply carries no tool call.
Returning a result differs per API as well, and the family notebooks run the whole loop:
[`01-openai-gpt/03`](../01-openai-gpt/03-tools-and-structured-output.ipynb),
[`14-openai-gpt-oss/01`](../14-openai-gpt-oss/01-gpt-oss-on-both-endpoints.ipynb)
section 5, and
[`02-anthropic-claude/02`](../02-anthropic-claude/02-thinking-tools-and-caching.ipynb).

In [9]:
STOCK_TOOL = {"type": "function", "function": {
    "name": "stock", "description": "Units in stock for a SKU.",
    "parameters": {"type": "object", "properties": {"sku": {"type": "string"}},
                   "required": ["sku"], "additionalProperties": False}}}
TOOL_PROMPT = "How many units of SKU A-100 are in stock? Use the stock tool."


def tool_call(model: str, api: str) -> tuple[str, dict, str]:
    """The tool one model asks for, the shape that API takes, and how the reply ended."""
    fn = STOCK_TOOL["function"]
    ask = [{"role": "user", "content": TOOL_PROMPT}]
    if api == "messages":
        reply = CLAUDE.messages.create(model=model, max_tokens=2000, messages=ask, tools=[
            {"name": fn["name"], "description": fn["description"], "input_schema": fn["parameters"]}])
        calls = [(b.name, b.input) for b in reply.content if b.type == "tool_use"]
        ended = reply.stop_reason
    else:
        client, model_id = client_for(model)
        if api == "responses":
            # Responses takes the function fields flat: nested it answers 400 Invalid 'tools'.
            reply = client.responses.create(model=model_id, input=TOOL_PROMPT, store=False,
                                            max_output_tokens=2000, tools=[{"type": "function", **fn}])
            calls = [(o.name, json.loads(o.arguments)) for o in reply.output if o.type == "function_call"]
            ended = reply.status
        else:
            reply = client.chat.completions.create(model=model_id, messages=ask, tools=[STOCK_TOOL],
                                                   max_completion_tokens=2000)
            calls = [(c.function.name, json.loads(c.function.arguments))
                     for c in reply.choices[0].message.tool_calls or []]
            ended = reply.choices[0].finish_reason
    if not calls:
        # No tool call is a result, not an empty one: name the stop, not a blank tool.
        raise RuntimeError(f"{model} asked for no tool on {api}; ended {ended}")
    return (*calls[0], ended)


for model, api in (("openai.gpt-oss-120b", "chat"), ("openai.gpt-5.4", "chat"),
                   ("openai.gpt-5.6-sol", "responses"), ("anthropic.claude-sonnet-5", "messages")):
    name, arguments, ended = tool_call(model, api)
    # The tool is a stand-in: the check reads the tool name given and the SKU asked for.
    print(f"{model:28} {api:10} ended {ended:11} -> {name}({json.dumps(arguments)})  "
          f"(asks stock for A-100: {name == 'stock' and arguments.get('sku') == 'A-100'})")


openai.gpt-oss-120b          chat       ended tool_calls  -> stock({"sku": "A-100"})  (asks stock for A-100: True)


openai.gpt-5.4               chat       ended tool_calls  -> stock({"sku": "A-100"})  (asks stock for A-100: True)


openai.gpt-5.6-sol           responses  ended completed   -> stock({"sku": "A-100"})  (asks stock for A-100: True)


anthropic.claude-sonnet-5    messages   ended tool_use    -> stock({"sku": "A-100"})  (asks stock for A-100: True)


## 7. Streaming on the three APIs

Each API streams a different event shape. The Responses API sends typed events with text
in `response.output_text.delta`, Chat Completions sends chunks with text in
`choices[0].delta.content`, and Messages sends `content_block_delta` events, which the
Anthropic SDK gathers into `text_stream`. The function below reads any of them and
returns the deltas with how the stream ended. How many deltas arrive is up to the model:
a short answer can come as one delta, so the cell prints the count next to the assembled
text.

Read how it ended before using the text. A complete answer is `stop` on Chat
Completions, `completed` on the Responses API and `end_turn` on Messages. Truncation at
the token cap arrives as `length`, `incomplete` with `incomplete_details.reason` of
`max_output_tokens`, and `max_tokens`, measured on the three models below with the cap
lowered to 20 and a prompt whose answer runs past it (`us-east-1`, 27 October 2026).
On the Responses API the truncated stream sends `response.incomplete` and no
`response.completed`, so `stream.get_final_response()` raises ``RuntimeError: Didn't
receive a `response.completed` event.`` (openai 2.50.0); the cell reads the terminal
event instead.

In [10]:
def stream_text(model: str, prompt: str, api: str, max_tokens: int = 300) -> tuple[list[str], str]:
    """The text deltas from any of the three APIs, and how the stream ended."""
    pieces, ended = [], "no terminal event"
    ask = [{"role": "user", "content": prompt}]
    if api == "messages":
        with CLAUDE.messages.stream(model=model, max_tokens=max_tokens, messages=ask) as stream:
            pieces = list(stream.text_stream)
            ended = stream.get_final_message().stop_reason
    elif api == "responses":
        client, model_id = client_for(model)
        with client.responses.stream(model=model_id, input=prompt, store=False,
                                     max_output_tokens=max_tokens) as stream:
            for event in stream:
                if event.type == "response.output_text.delta":
                    pieces.append(event.delta)
                elif event.type in ("response.completed", "response.incomplete", "response.failed"):
                    # A truncated stream sends response.incomplete and no completion event, so
                    # get_final_response() raises there; read the terminal event instead.
                    details = event.response.incomplete_details
                    ended = event.response.status + (f" ({details.reason})" if details else "")
    else:
        client, model_id = client_for(model)
        for chunk in client.chat.completions.create(
                model=model_id, messages=ask, max_completion_tokens=max_tokens, stream=True):
            if chunk.choices and chunk.choices[0].delta.content:
                pieces.append(chunk.choices[0].delta.content)
            if chunk.choices and chunk.choices[0].finish_reason:
                ended = chunk.choices[0].finish_reason
    return pieces, ended


for model, api in (("google.gemma-4-31b", "responses"), ("qwen.qwen3-32b", "chat"),
                   ("anthropic.claude-haiku-4-5", "messages")):
    pieces, ended = stream_text(model, "Count from 1 to 10, separated by commas.", api)
    text = "".join(pieces).strip()
    # The counting prompt has no single right wording, so the check reads the numbers
    # out of the assembled text rather than comparing the text itself.
    numbers = [int(n) for n in re.findall(r"\d+", text)]
    print(f"{model:28} {api:10} {len(pieces):3} deltas, ended {ended} -> {text}"
          f"  (counts 1 to 10: {numbers == list(range(1, 11))})")

google.gemma-4-31b           responses   29 deltas, ended completed -> 1, 2, 3, 4, 5, 6, 7, 8, 9, 10  (counts 1 to 10: True)


qwen.qwen3-32b               chat         1 deltas, ended stop -> 1, 2, 3, 4, 5, 6, 7, 8, 9, 10  (counts 1 to 10: True)


anthropic.claude-haiku-4-5   messages     4 deltas, ended end_turn -> 1, 2, 3, 4, 5, 6, 7, 8, 9, 10  (counts 1 to 10: True)


## 8. LLM gateways

Gateways such as LiteLLM reach Bedrock through the same OpenAI-compatible URLs. Give
each model its own `api_base`, because the path depends on the family:

```yaml
model_list:
  - model_name: frontier
    litellm_params:
      model: openai/us.openai.gpt-5.6-sol
      api_base: https://bedrock-runtime.us-east-1.amazonaws.com/openai/v1
      api_key: os.environ/BEDROCK_API_KEY
  - model_name: open-weight
    litellm_params:
      model: openai/qwen.qwen3-32b
      api_base: https://bedrock-mantle.us-east-1.api.aws/v1
      api_key: os.environ/BEDROCK_API_KEY
```

The key expires within 12 hours and cannot be renewed, so mint a new one on a schedule,
well inside that lifetime, and hand it to the gateway in whatever way your deployment
reloads its environment:

```bash
export BEDROCK_API_KEY="$(python -c 'from aws_bedrock_token_generator import provide_token; print(provide_token(region="us-east-1"))')"
```

## 9. What you gain and what you lose

As of September 2026:

| Moving from | You gain | You lose or must change |
|---|---|---|
| The OpenAI API | IAM authentication, Projects, data retention controls, service tiers, models from several providers, and web search on the hosted GPT models | OpenAI model IDs, sampling parameters on some models, and `max_tokens` on GPT-5.6's Chat Completions |
| The Anthropic API | IAM authentication, Workspaces, and the same Claude models on `bedrock-runtime` through Converse | `temperature` and `top_p` on newer Claude models, which refuse them as deprecated, and a forced `tool_choice` on Sonnet 5.5, Opus 5.5, Fable 5.1 and Fable 5.2, which answer 400 `tool_choice: type "tool" and "any" are not supported for this model.`, so leave the choice on auto or send the schema in `output_config.format`, which the eleven models in that notebook's table accepted, ten of them on 20 October 2026 and Fable 5.2 on 10 November 2026 (`us-east-1`, [`02-anthropic-claude/01`](../02-anthropic-claude/01-messages-api-core.ipynb)) |
| `bedrock-runtime`, to `bedrock-mantle` | The bare `/v1` path, models only there such as Gemma 4 and Grok 4.3, bare model IDs with no profile, server-side tools such as web search, `background=true`, and the control plane for Projects, data retention, files and fine-tuning | Converse, cross-Region inference profiles, Provisioned Throughput, batch inference, and Guardrails, whose header every `bedrock-mantle` API ignores ([`03-production-hardening`](03-production-hardening.ipynb)) |

## 10. A migration checklist

1. Map each model name to a Bedrock model ID, per endpoint.
2. Point each model at its URL path; one base URL does not cover every family.
3. Remove shared sampling defaults, and add `temperature` or `top_p` back per model
   after testing.
4. Send `max_completion_tokens` on Chat Completions.
5. Check that each model serves the API your code calls.
6. Shape each `tools` list for the API you call: Messages takes
   `input_schema` with no function wrapper (section 6).
7. Port `embeddings.create` separately: the OpenAI-compatible embeddings route is
   served on both endpoints and has no model behind it, so embeddings go through
   `bedrock-runtime` `InvokeModel` with the provider's body (Things that differ).
8. Send `service_tier` only where the model accepts it.
9. Set `store` explicitly on the Responses API, which keeps responses for 30 days by
   default. It governs whether you can retrieve and continue from a response rather
   than what is retained, and retention follows the mode
   ([`00-foundations/02`](../00-foundations/02-governance-projects-and-retention.ipynb)).
10. Replace `client.models.list()` on `bedrock-runtime`, which does not serve it,
   with `ListFoundationModels` and `ListInferenceProfiles` (section 1).
11. Replace static API keys with short-term keys minted from IAM, renewed before they
   expire.
12. Retry 429 and 5xx with backoff, and fail fast on other 4xx.
13. Set a client timeout on every call.
14. Read how a stream ended before using its text; each API reports it differently.
15. Check the values in structured output.
16. Point dashboards for `bedrock-mantle` traffic at the `AWS/BedrockMantle` namespace.
17. Give each workload a tagged Project for cost attribution.
18. Confirm that every model you depend on is in your Region.

## Things that differ

What trips people up here, and what to do about it.

| Behaviour | What you see | What to do |
|---|---|---|
| OpenAI model names | 404 `The model 'gpt-4o' does not exist` | Map each name to a Bedrock model ID (section 3). |
| The wrong URL path | 400 ``model `openai.gpt-oss-120b` isn't supported on this route`` from a model that is served on the other Chat Completions path, which was the wording for each of the seven non-Claude models tried, or 400 `The model 'anthropic.claude-sonnet-5' does not support the '/v1/chat/completions' API` from the Claude models, which serve neither chat path and name `/v1/chat/completions` even for a request sent to `/openai/v1/chat/completions` (`us-east-1`, 12 November 2026; [`01-choosing-a-model-and-api`](01-choosing-a-model-and-api.ipynb) measures both wordings across the models it surveys) | Keep one client per path (section 4). The Claude wording names `/v1/chat/completions` whichever chat path you called, so read it as the route the service matched rather than the one you called; the Responses refusal further down this table names the path you called. Read 404 `The model '…' does not exist` as a question about the model ID instead. |
| `client.models.list()` | 404 `<UnknownOperationException/>` on `bedrock-runtime`, raised as `openai.NotFoundError` | The OpenAI-compatible Models API is served on `bedrock-mantle` `/v1`; on `bedrock-runtime` use `ListFoundationModels` or `ListInferenceProfiles` (section 1). |
| `max_tokens` on GPT-5.6 | 400 `Unsupported parameter: 'max_tokens' is not supported with this model.` | Send `max_completion_tokens` on Chat Completions. |
| Shared sampling defaults | A 400 that names the parameter | Send none by default, or drop what the 400 names (section 5). |
| A function tool | 400 ``tool type 'function' is not supported for this model`` on Messages, or 400 `Function tools with reasoning_effort are not supported for gpt-5.6-sol in /v1/chat/completions.` on that model's Chat Completions, with no `reasoning_effort` sent, or 400 ``invalid request body: Invalid 'tools': missing field `name` `` when the nested Chat Completions tool shape is sent to the Responses API, and ``missing field `function` `` for the reverse | Send the tool as `{"name", "description", "input_schema"}` for Claude, nested under `"function"` on Chat Completions and flat on the Responses API, and send `openai.gpt-5.6-sol` a tool on `/openai/v1/responses`, or on Chat Completions with `reasoning_effort="none"` (section 6). |
| The Responses API on a path that does not serve it | 400 `The model '…' does not support the '/v1/responses' API`, which is about the route rather than the model: `openai.gpt-5.6-sol` returns it on `/v1/responses` and answers 200 on `/openai/v1/responses`, while `openai.gpt-oss-120b` is the other way round, and `anthropic.claude-sonnet-5` and `anthropic.claude-haiku-4-5` return it on either path, each naming the path called (`us-east-1`, 12 November 2026) | Send the Responses API to the path `api_prefix` gives for that model (section 4), or use Chat Completions, which the families in section 4 other than Claude serve. |
| A `bedrock-mantle` ID on `bedrock-runtime` | `Invocation of model ID … with on-demand throughput isn’t supported`, or `The provided model identifier is invalid` | Use the runtime ID or its inference profile ([`00-foundations/01`](../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb)). |
| `embeddings.create` | On `bedrock-runtime`'s `/openai/v1` path the embeddings route answers, and serves no model: `amazon.titan-embed-text-v2:0` and the chat profile `us.openai.gpt-5.6-sol` both return 404 `The model doesn't exist or doesn't support this API. Retry your request with a different model ID.` with `code: model_not_found`, while an ID form the endpoint does not take, `openai.gpt-oss-120b` without a profile prefix or `us.amazon.titan-embed-text-v2:0` with one, returns 400 `The provided model identifier is invalid.` with `code: validation_error`. The OpenAI SDK raises those as `NotFoundError` and `BadRequestError`. A client pointed at the bare `/v1` path on that host meets the other failure instead, HTTP 200 with a Coral `UnknownOperationException` body that the SDK turns into `ValueError: No embedding data received` (section 4). On `bedrock-mantle` `/v1/embeddings` exists but is model-gated, and that catalogue carries no embedding model in `us-east-1`, so a chat model returns 400 `The model '…' does not support the '/v1/embeddings' API` and an embedding model 404 `The model '…' does not exist` (`us-east-1`, 19 October 2026) | Port embeddings to `bedrock-runtime` `InvokeModel` with the provider's own body, such as `amazon.titan-embed-text-v2:0` with `{"inputText": "..."}`, which answers 200 with a 1024-dimension `embedding`. Converse does not serve those models either: the same model refuses it with `This action doesn't support the model that you provided. Try again with a supported text or chat model.` `ListFoundationModels` filtered on output modality `EMBEDDING` lists them (section 3). |

## Next

- Which model and API, measured across families:
  [`01-choosing-a-model-and-api`](01-choosing-a-model-and-api.ipynb)
- Before production traffic: [`03-production-hardening`](03-production-hardening.ipynb)
- Projects and data retention:
  [`00-foundations/02`](../00-foundations/02-governance-projects-and-retention.ipynb)